# Project Demo

**CSCI 222 Foundations of Large Language Models**

_Eric Ordonez, 5/16/2026_

If running this notebook in Colab and connected to Google Drive, specify its location within Google Drive here, relative to MyDrive.

In [ ]:
GDRIVE_PATH = 'HES/CSCI_222/Project'

In [ ]:
import json
import os
import re
import sys
import time
import warnings
from pathlib import Path

import bm25s
import faiss
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import seaborn as sns
import torch
import umap
from dotenv import load_dotenv
from huggingface_hub import login
from sentence_transformers import CrossEncoder, SentenceTransformer
from tabulate import tabulate
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

from src.evaluation import init_generator, score, summarize
from src.rankings import (get_bm25_ranking, get_faiss_ranking,
                          get_rrf_ranking, rerank)

In [ ]:
# Mount Google Drive if running in Colab.
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
    PROJECT_ROOT = Path('/content/drive/MyDrive') / GDRIVE_PATH
else:
    PROJECT_ROOT = Path('.')

os.chdir(PROJECT_ROOT)
DATA_DIR = PROJECT_ROOT / 'data'
DATA_DIR.mkdir(exist_ok=True)
RESULTS_DIR = PROJECT_ROOT / 'results'
RESULTS_DIR.mkdir(exist_ok=True)

# Use a GPU if available.
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USING_GPU = DEVICE.type == 'cuda'
print(f'Device: {DEVICE}')
if USING_GPU:
    print(f"  GPU:  {torch.cuda.get_device_name(0)}")
    print(f"  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# Load the project .env.
success = load_dotenv(PROJECT_ROOT / '.env')
assert success, "Failed to load .env. Check path or Drive mount."

# Load the Hugging Face token.
HF_TOKEN = os.getenv('HF_TOKEN')
if not HF_TOKEN:
    raise RuntimeError('Hugging Face token not found in .env.')
login(token=HF_TOKEN)

## Data loading

A JSON file of the 10k paper subsample is already provided in `data`, so no OpenAlex API key is needed to run this demo.

In [ ]:
papers_10k_json = DATA_DIR / 'papers_10k.json'
with open(papers_10k_json, 'r') as f:
    papers = json.load(f)
    print(f"Loaded {len(papers):,} papers from {papers_10k_json}.")

df = pd.DataFrame(papers)
df.head()

`id` and `doi` are useful for unique identification, but they are not needed in this pipeline.
The actual retrieval-augmented generation depends only upon `title` and `abstract`.
The other metadata (`year`, `authors`, and `journal`) are for user context.

There are potentially useful ways to use these data to augment generation ("Find results only from top 5 journals and published since 2000"), but those are beyond the scope of this project.

## Data exploration

The abstract is the primary source of semantic content, and we will be encoding its content with a Sentence Transformer.


In [ ]:
word_counts = df['abstract'].apply(lambda x: len(x.split()))

_, ax = plt.subplots()
sns.histplot(np.log1p(word_counts) / np.log(10), ax=ax)
ax.set_xlabel('Number of words (log10)')
ax.set_ylabel('Count')
ax.set_title('Abstract word count')
plt.tight_layout()
plt.show()

The word count distribution is heavily right skewed.
The average is around 100 words, but there is a large propotion of abstracts that tend closer to 1,000 words.
This variation is one reason underpinning why we cannot use lexical similarity between papers alone as a measure of relevance.

In [ ]:
_, ax = plt.subplots()

sns.barplot(df['topic'].value_counts(ascending=False), ax=ax)
ax.set_xticks([])
ax.set_xlabel('Paper topics (not labeled here)')
ax.set_ylabel('Count')
ax.set_title('Paper topics')
plt.tight_layout()
plt.show()

There are 87 topics in the subsample (not labeled here), with a significant number of papers belonging to more than of half of them.
This introduces semantic variation that must be accounted for in the retrieval system, too.
Even within economics and finance, technical vocabulary can mean different things from one subfield to another.
An instrument in finance is distinct from an instrument in econometrics.

## Embeddings

We use the SPECTER2 base model for vector embeddings.

In [ ]:
model_name = 'allenai/specter2_base'
model = SentenceTransformer(model_name, device=DEVICE)

SPECTER was trained on title/abstract pairs.
It explicitly expects its inputs to be of the form `{title} + [SEP] + {abstract}`, where `[SEP]` is a separation token.

The embedding dimension is 768, so the output here is a tensor of shape(10,000, 768).
We also normalize the embeddings for the vector indexing in the next section.

In [ ]:
specter_embeddings_10k_npy = DATA_DIR / 'specter_embeddings_10k.npy'
if specter_embeddings_10k_npy.exists():
    embeddings = np.load(specter_embeddings_10k_npy)
    print(f"Loaded SPECTER embeddings from {specter_embeddings_10k_npy}.")
else:
    texts = [p['title'] + " [SEP] " + p['abstract'] for p in papers]
    embeddings = model.encode(
        texts,
        batch_size=64,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True
    )
    embeddings = embeddings.astype('float32')
    np.save(specter_embeddings_10k_npy, embeddings)
    print(f"Saved SPECTER embeddings to {specter_embeddings_10k_npy}.")

## Rankings

A RAG model receives a query, computes relevancy scores for documents in the retrieval corpus, and returns the top ranked documents.
There are four ranking methods that we compare qualitatively:

- FAISS (Facebook AI Similarity Search)
- BM25 (Best Matching 25)
- RRF (Retrieval Rank Fusion)
- Cross-encoded reranking

The FAISS and BM25 rankings are fused into a single RRF ranking, which is then reranked by a cross-encoder.
That reranking is the actual ranking that the pipeline uses.

For this demo, we will test the same query and retrieve the top 10 results from each ranker.

In [ ]:
# Test a query.
query = "effects of minimum wage on unemployment"
top_k = 10
k = 50

### FAISS

In [ ]:
faiss_index_file = DATA_DIR / 'specter_index.faiss'
if faiss_index_file.exists():
    index = faiss.read_index(str(faiss_index_file))
    print(f"Loaded FAISS index from {faiss_index_file}.")
else:
    index = faiss.IndexFlatIP(embeddings.shape[1])
    index.add(embeddings)
    faiss.write_index(
        index,
        str(DATA_DIR / 'specter_index.faiss')
    )
    print(f"Saved FAISS index to {faiss_index_file}.")

In [ ]:
faiss_ranking = get_faiss_ranking(query, model, index, k=k)
faiss_top_k   = [(idx, papers[idx]) for idx in faiss_ranking[:top_k]]

### BM25

In [ ]:
bm25s_index_dir = DATA_DIR / 'bm25s'
if bm25s_index_dir.exists():
    retriever = bm25s.BM25.load(bm25s_index_dir)
    print(f"Loaded retriever index and corpus from {bm25s_index_dir}.")
else:
    corpus = [p['title'] + " " + p['abstract'] for p in papers]
    corpus_tokens = bm25s.tokenize(corpus)
    retriever = bm25s.BM25()
    retriever.index(corpus_tokens)
    retriever.save(bm25s_index_dir)
    print(f"Saved retriever index and corpus to {bm25s_index_dir}.")

In [ ]:
bm25_ranking = get_bm25_ranking(query, retriever, k=k)
bm25_top_k   = [(idx, papers[idx]) for idx in bm25_ranking[:top_k]]

### RRF

In [ ]:
rrf_ranking = get_rrf_ranking(faiss_ranking, bm25_ranking)
rrf_top_k   = [(idx, papers[idx]) for idx in rrf_ranking[:top_k]]

### Reranking

Given a list of documents and a query, the cross-encoder simultaneously attends to both the query and each document to compute relevance scores.
Contrast this to the bi-encoder in SPECTER, which encoded the query and documents independently.

In [ ]:
ce_name = 'cross-encoder/ms-marco-MiniLM-L-6-v2'
cross_encoder = CrossEncoder(ce_name, device=DEVICE)

We can rerank the fused ranking according to the scores to get a final ranking to use for retrieval.

In [ ]:
candidate_papers = [(i, papers[i]) for i in rrf_ranking]
reranking = rerank(cross_encoder, query, candidate_papers)

In [ ]:
reranked_top_k = [(idx, papers[idx]) for idx in reranking[:top_k]]
print(tabulate(
    [
        [rank, p['title'], p['year'], p['journal']]
        for rank, (_, p) in enumerate(reranked_top_k, start=1)
    ],
    headers=['RANK', 'TITLE', 'YEAR', 'JOURNAL'],
    tablefmt='grid',
    maxcolwidths=[None, 40, None, 20]
))

Retrieval quality passes the eye test: we asked about the effect of the minimum wage on unemployment, and each retrieved paper's title features both "minimum wage" and "(un)employment."

Did reranking, both by fusion and cross encoding, do anything?
We can see so by comparing the rankings.
We restrict ourselves to the top 5 for compactness.

In [ ]:
print(tabulate(
    zip(range(1, 6),
        [p['title'] for _, p in faiss_top_k[:5]],
        [p['title'] for _, p in bm25_top_k[:5]],
        [p['title'] for _, p in rrf_top_k[:5]],
        [p['title'] for _, p in reranked_top_k[:5]]),
    headers=['RANK', 'FAISS', 'BM25', 'RRF', 'RERANKED'],
    tablefmt='grid',
    maxcolwidths=[None, 16, 16, 16, 16]
))

Although the titles are clearly relevant, the next step is to assess whether the papers themselves are useful by synthesizing and summarizing their abstracts.

## Evaluation

We use an LLM for the evaluation stage.

In [ ]:
llm_name = 'meta-llama/Llama-3.1-8B-Instruct'
llm = AutoModelForCausalLM.from_pretrained(
    llm_name,
    torch_dtype='auto',
    device_map='auto'
)
tokenizer = AutoTokenizer.from_pretrained(
    llm_name,
    clean_up_tokenization_spaces=False
)

In [ ]:
generator = pipeline(
    'text-generation',
    model=llm,
    tokenizer=tokenizer,
)

# Set module generator.
init_generator(generator)

### Summarization

In [ ]:
summary = summarize(query, reranked_top_k[:5])
print(summary)

### Relevance scores

The LLM synthesizes the abstracts into something that may be useful, but even if the summary is well written, the adage "garbage in, garbage out" applies as always.
We have no measure of whether the retrieved features are relevant in the first place, so we use an LLM as a judge of relevance.

It is instructed to score an retrieved paper's relevance to the query on a scale from 0 to 3:

- 0 = Not relevant
- 1 = Tangentially related
- 2 = Relevant
- 3 = Highly relevant

Each ranking's mean relevance to the query is computed here.

In [ ]:
all_papers = faiss_top_k + bm25_top_k + rrf_top_k + reranked_top_k
all_scores = score(query, all_papers)
scores = {
    'faiss':    all_scores[0:top_k],
    'bm25':     all_scores[top_k:2*top_k],
    'rrf':      all_scores[2*top_k:3*top_k],
    'reranked': all_scores[3*top_k:4*top_k]
}

In [ ]:
mean_scores = {
    k: np.mean([result['score'] for result in scores[k]])
    for k in scores.keys()
}
print(tabulate(
    list(mean_scores.items()),
    headers=['method', 'mean score']))

## Embedding visualization

Is there other information from the OpenAlex snapshots that we could use to enhance RAG?
Even though we cannot pull the full text of every paper in our corpus, we still have metadata that suggests more underlying structure that we could exploit in further work.

Consider paper topics.
We project the SPECTER embeddings into 2D space via UMAP and identify them by their corresponding paper's topics.
Since there are 87 topics in the subample, we limit the projection to papers in the top ten most numerous topics.

In [ ]:
reducer = umap.UMAP(n_components=2, metric='cosine', n_jobs=1, random_state=222)
coords = reducer.fit_transform(embeddings)
labels = [p['topic'] if p['topic'] is not None else 'Unknown' for p in papers]

In [ ]:
df_umap = pd.DataFrame({
    'x': coords[:,0],
    'y': coords[:,1],
    'topic': labels
})

What exactly are these topics?

In [ ]:
topics = pd.Series([p['topic'] for p in papers])
print(f"Number of unique paper topics: {topics.nunique()}")

In [ ]:
top10_topic_counts = topics.value_counts(ascending=False)[:10]
top10_topic_counts

We plot the 2D projection below.

In [ ]:
top10_topics = top10_topic_counts.index
df_umap_clipped = df_umap[(df_umap['x'] >= 5) & (df_umap['y'] >= 0)]

_, ax = plt.subplots(figsize=(12.8, 4.8))
sns.scatterplot(
    # Filtering for x >= 5 is specific to this seed.
    data=df_umap_clipped[df_umap_clipped['topic'].isin(top10_topics)],
    x='x',
    y='y',
    hue='topic',
    s=3,
    alpha=0.67,
    ax=ax
)
ax.set_xlabel('UMAP 1')
ax.set_ylabel('UMAP 2')
ax.set_title('UMAP projection of the dense embeddings')
ax.legend(
    title='Topic',
    bbox_to_anchor=(1.02, 1),
    loc='upper left',
    borderaxespad=0,
    markerscale=6
)
plt.tight_layout()
plt.show()